# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 19 · One matched learned-representation comparison

Two models, one reused chronological fold. Architecture, initial weights, optimizer and exposure match. Only terminal versus observed pair histories differ. The existing tree is a separate comparator, not a feature-only attribution.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT=Path('/home/sagemaker-user/nfl_feature_round8')
OUT=Path('/home/sagemaker-user/nfl-feature-round8-results')
PY=Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract this kit first.')
sys.path.insert(0,str(KIT))
import visuals
pio.renderers.default='plotly_mimetype'
def run(stage,*options):
    p=subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage,*options],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='')
        code=p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve checkpoints and return the report; do not change settings.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


## CPU runtime and engineering checks
Offline reuse first. Only a missing cached-package error permits the single `runtime --online` setup command in START_HERE.md. Never install into the notebook environment.

In [ ]:
run('runtime')

## Training-only throughput gate
Sixteen disposable steps are not scientific results. If the profile fails its fixed budget, stop here and return the report.

In [ ]:
run('profile')

## Two fixed-exposure arms
No validation checkpoint selection and no additional folds. Each stage retains resumable checkpoints.

In [ ]:
run('train','--arm','terminal')

In [ ]:
run('train','--arm','history')

## Evaluate only the completed pair

In [ ]:
run('evaluate')
s=json.loads((OUT/'summary.json').read_text())
print(json.dumps({'metrics':s['metrics'],'contrast':s['contrast'],'ready_for_later_fold':s['ready_for_later_fold']},indent=2))

In [ ]:
show(visuals.learning(OUT),'training_objective')
show(visuals.metrics(OUT),'matched_metrics')
show(visuals.interval(OUT),'paired_game_interval')
show(visuals.horizons(OUT),'horizon_errors')

## Fresh-process verification and export
Replay may not fit missing or incomplete models. Even a passing gate does not automatically start another fold.

In [ ]:
run('replay')
run('report')
print('Download:', OUT/'nfl_feature_round8_report.zip')

Save both notebooks and stop the existing space. Return the aggregate report. No new Kaggle score or production promotion is claimed.